In [1]:
import json
import socket
import threading

In [2]:
class UdpEventBuffer(threading.Thread):

    def __init__(self, host, port):
        """初始化操作"""
        super().__init__(daemon=True)
        self.host = host
        self.port = int(port)
        self._sock  = None
        self._stop  = threading.Event()
        self._lock  = threading.Lock()
        self._queue = []

    def run(self):
        """创建进程监听"""
        #                 使用IPV4         使用UDP
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        #            不是TCP/UDP协议层级  允许地址复用
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        #       绑定ip地址  绑定端口
        s.bind((self.host, self.port))
        self._sock = s
        # 阻塞接受最多等待0.25秒就抛出timeout异常
        s.settimeout(0.25)
        # 接收到结束信号才退出循环, 其中结束信号由stop函数设置
        while not self._stop.is_set():
            try:
                # 阻塞线程, 释放GIL锁
                data, _ = s.recvfrom(65535)
            except socket.timeout:
                # 若超时, 则继续
                continue
            except OSError:
                # 系统错误
                break
            try:
                # 使用json转化报文为utf-8格式
                obj = json.loads(data.decode("utf-8", errors="replace"))
                # 判断报文是否为dict格式
                if not isinstance(obj, dict):
                    continue
            except Exception:
                continue
            # 若锁被占用, 则会进入等待状态
            with self._lock:
                # 尝试拿到锁并往变量_queue中塞入报文
                self._queue.append(obj)

        try:
            s.close()
        except Exception:
            pass
        self._sock = None

    def stop(self):
        self._stop.set()

    def drain_events(self):
        # 若锁被占用, 则会进入等待状态
        with self._lock:
            # 尝试拿到锁并清空_queue
            evs = self._queue
            self._queue = []
        return evs

In [3]:
if __name__ == "__main__":
    udp_gain = UdpEventBuffer("0.0.0.0", 28115)
    udp_gain.start()

    udp_gain.drain_events()
    # udp_gain.stop()

In [4]:
def _compute_reward_from_events(_reward, events, last_hp, enemy_dict):
    terminated = False
    message_dict = events
    # 如果udp报文类型为场景变化, 则结束游戏
    if events['type'] == 'scene_changed':
        _alive = False
        terminated = True
        last_hp = 9
    else:
        # 若不为空则计算奖励
        ## 首先计算角色血量变化奖励, 扣血减去 扣血量*_reward_max/max_hp, 加血加上 加血量*10
        _reward += (message_dict["player"]["hp"] - last_hp) * 10
        last_hp = message_dict["player"]["hp"]

        ## 然后计算攻击敌人奖励, 每打掉1点血+1奖励
        for enemy in message_dict["enemies"]:
            # 如果这个敌人之前没有记录, 则加入存储中
            if enemy_dict.get(enemy['id']) == None:
                enemy_dict[enemy['id']] = {
                    "name":  enemy['name'],
                    "hp":    enemy['hp'],
                    "hpMax": enemy['hp']
                }
            # 如果这个敌人之前存在记录, 则计算奖励
            else:
                # 如果敌人的hp出现上升, 代表之前没有捕捉到敌人的血量初始化
                if enemy_dict[enemy['id']]['hp'] < enemy['hp']:
                    # 血量发生变化说明之前攻击过敌人
                    _reward += 2.1
                    enemy_dict[enemy['id']]['hp']    = enemy['hp']
                    enemy_dict[enemy['id']]['hpMax'] = enemy['hp'] + 21
                # 如果敌人的hp不变或下降, 则计算奖励
                else:
                    _reward += (enemy_dict[enemy['id']]['hp'] - enemy['hp']) * 0.1
                    enemy_dict[enemy['id']]['hp'] = enemy['hp']

    return _reward, terminated, last_hp, enemy_dict

In [7]:
_reward = 0
last_hp = 9
enemy_dict = {}
while True:
    event_list = udp_gain.drain_events()
    for events in event_list:
        new_reward, finished, last_hp, enemy_dict = _compute_reward_from_events(_reward, events, last_hp, enemy_dict)
        print("-----")
        print(events)
        print(new_reward - _reward, finished)
        _reward = new_reward

-----
{'type': 'scene_changed', 'ts': 1769054629832, 'seq': 6, 'scene': 'GG_Workshop', 'scenes': ['GG_Workshop', 'GG_Mantis_Lords'], 'snapshot': {'player': {'id': 83368, 'hp': 9, 'hp_blue': 0, 'hp_max': 9}, 'enemies': [{'id': 116046, 'name': 'Mantis Lord', 'hp': 210}, {'id': 115804, 'name': 'Mantis Lord S1', 'hp': 160}, {'id': 116064, 'name': 'Mantis Lord S2', 'hp': 160}]}}
0 True
-----
{'type': 'hp_update', 'ts': 1769054638525, 'seq': 7, 'scene': 'GG_Workshop', 'scenes': ['GG_Workshop', 'GG_Mantis_Lords'], 'player': {'id': 83368, 'hp': 9, 'hp_blue': 0, 'hp_max': 9}, 'enemies': [{'id': 116046, 'name': 'Mantis Lord', 'hp': 379}, {'id': 115804, 'name': 'Mantis Lord S1', 'hp': 160}, {'id': 116064, 'name': 'Mantis Lord S2', 'hp': 160}], 'changed': {'ids': [116046]}}
1 False
-----
{'type': 'hp_update', 'ts': 1769054638980, 'seq': 8, 'scene': 'GG_Workshop', 'scenes': ['GG_Workshop', 'GG_Mantis_Lords'], 'player': {'id': 83368, 'hp': 9, 'hp_blue': 0, 'hp_max': 9}, 'enemies': [{'id': 116046, 'n

KeyboardInterrupt: 

In [8]:
udp_gain.stop()